<a href="https://colab.research.google.com/github/ZeEshAn-boot/AI-Product-Recommendation-System/blob/main/Face_detection_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from torchvision import datasets, transforms
from PIL import Image

import cv2

import time

In [23]:
#Configuration

DATA_DIR = "./data"
IMG_SIZE = 64
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001
VAL_SPLIT = 0.2
MODEL_PATH = "./model.pth"

In [24]:
device = torch.device("cuda")
print(device)

cuda


In [25]:
class CNN(nn.Module):
  def __init__(self, num_classes):
    super().__init__()
    self.features = nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, padding=1),
        nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(), nn.MaxPool2d(2)
    )

    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(4096,128),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(128,num_classes)
    )

  def forward(self, x):
    x = self.features(x)
    x = self.classifier(x)
    return x

In [26]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

In [27]:
import zipfile
import os
zip_path = '/content/emotion_detector-data.zip'
extraction_path = './'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

In [28]:
import os
DATA_DIR = './processed_data'
print(f"Attempting to load data from: {DATA_DIR}")
print("Contents of current directory:", os.listdir('.'))
full_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transform)
len(full_dataset)

Attempting to load data from: ./processed_data
Contents of current directory: ['.config', '.ipynb_checkpoints', 'emotion_detector-data.zip', 'processed_data', 'vedio.mp4', 'sample_data']


49779

In [29]:
class_names = full_dataset.classes
num_class = len(class_names)
class_names

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

In [30]:
val_count = int(len(full_dataset)*VAL_SPLIT)
train_count = len(full_dataset) - val_count

train, val = random_split(full_dataset, [train_count, val_count])

In [31]:
train_count, val_count

(39824, 9955)

In [32]:
train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True, num_workers=12, pin_memory=True)
val_loader = DataLoader(val, batch_size=BATCH_SIZE, shuffle=True, num_workers=12, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = CNN(num_classes=num_class).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

Using device: cpu


In [34]:
def train():
  best_acc = 0.0

  for epoch in range(1, EPOCHS + 1):
    model.train()
    avg_loss = 0.0
    avg_correct = 0
    total = 0
    t0 = time.time()
    for images,labels in train_loader:
      images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

      optimizer.zero_grad()
      outputs = model(images)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()

      avg_loss += loss.item()*images.size(0)
      preds = outputs.argmax(dim=1)
      avg_correct += (preds == labels).sum().item()
      total += images.size(0)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
      for images,labels in val_loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item()*images.size(0)
        preds = outputs.argmax(dim=1)
        val_correct += (preds == labels).sum().item()
        val_total += images.size(0)

    avg_loss /= total
    avg_acc = avg_correct/total
    val_loss /= val_total
    val_acc = val_correct/val_total
    t1 = time.time()
    elapsed = t1-t0
    print(f"EPOCH {epoch}/{EPOCHS}  train_loss={avg_loss:.3f} train_acc={avg_acc:.3f}  val_loss={val_loss:.3f} val_acc={val_acc:.3f}  time={elapsed:.1f}s")

    if val_acc>best_acc:
      best_acc = val_acc
      torch.save({
          "model_state" : model.state_dict(),
          "class_names" : class_names,
          "img_size" : IMG_SIZE
      }, MODEL_PATH)

      print(f"Saving the model with the best validation accuracy : {best_acc:.3f} to {MODEL_PATH}")

  print("Training completed")

In [38]:
def inference():
  import matplotlib.pyplot as plt
  from IPython.display import Video, display
  from PIL import Image as PILImage # Use alias to avoid conflict with Image from torchvision
  import imageio # Import imageio for video creation

  checkpoint = torch.load(MODEL_PATH,map_location=device)
  model.load_state_dict(checkpoint["model_state"])
  model.to(device)
  model.eval()
  class_names = checkpoint["class_names"]

  print("Starting video inference...")
  webcam = cv2.VideoCapture("/content/vedio.mp4")
  if not webcam.isOpened():
    print("Webcam error: Could not open video file. Make sure '/content/vedio.mp4' exists.")
    return

  # Get video properties for output video
  fps = webcam.get(cv2.CAP_PROP_FPS)
  width = int(webcam.get(cv2.CAP_PROP_FRAME_WIDTH))
  height = int(webcam.get(cv2.CAP_PROP_FRAME_HEIGHT))

  output_frames = []
  frame_count = 0
  # Process a limited number of frames for demonstration if the video is long
  # You can adjust or remove max_frames_to_process to process the entire video
  max_frames_to_process = 100 # Process a short segment of the video (e.g., 100 frames)

  while True:
    ret, frame = webcam.read()

    if not ret:
      print("End of video stream or error reading frame.")
      break

    if frame_count >= max_frames_to_process:
        print(f"Processed {max_frames_to_process} frames. Stopping for demonstration.")
        break

    rbg = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resize_img = PILImage.fromarray(rbg).resize((IMG_SIZE, IMG_SIZE))
    tensor = transforms.functional.to_tensor(resize_img)
    tensor = transforms.functional.normalize(tensor,[0.485,0.456,0.406],[0.229,0.224,0.225])
    tensor = tensor.unsqueeze(0).to(device)

    with torch.no_grad():
      out = model(tensor)
      probs = torch.nn.functional.softmax(out, dim=1)
      top_prob, prob_idx = torch.max(probs, dim=1)
      label = class_names[prob_idx.item()]
      conf = top_prob.item()

    cv2.putText(frame, f"{label} {conf:.2f}", (10,30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,0), 2)

    # Store the processed frame (in RGB format for imageio)
    output_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    frame_count += 1

  webcam.release()

  output_video_path = '/content/output_emotion_video.mp4'
  print(f"Saving processed video to {output_video_path}")

  if output_frames:
      imageio.mimwrite(output_video_path, output_frames, fps=fps, quality=8) # Lower quality for faster saving/smaller file
      print("Displaying the processed video:")
      display(Video(output_video_path, embed=True, width=width, height=height))
  else:
      print("No frames were processed to create a video.")


In [36]:
train()

EPOCH 1/10  train_loss=1.475 train_acc=0.429  val_loss=1.209 val_acc=0.536  time=196.8s
Saving the model with the best validation accuracy : 0.536 to ./model.pth
EPOCH 2/10  train_loss=1.195 train_acc=0.545  val_loss=1.102 val_acc=0.576  time=200.8s
Saving the model with the best validation accuracy : 0.576 to ./model.pth
EPOCH 3/10  train_loss=1.112 train_acc=0.577  val_loss=1.050 val_acc=0.596  time=206.2s
Saving the model with the best validation accuracy : 0.596 to ./model.pth
EPOCH 4/10  train_loss=1.051 train_acc=0.601  val_loss=1.008 val_acc=0.610  time=204.7s
Saving the model with the best validation accuracy : 0.610 to ./model.pth
EPOCH 5/10  train_loss=1.008 train_acc=0.617  val_loss=0.999 val_acc=0.615  time=198.5s
Saving the model with the best validation accuracy : 0.615 to ./model.pth
EPOCH 6/10  train_loss=0.980 train_acc=0.625  val_loss=0.983 val_acc=0.629  time=204.0s
Saving the model with the best validation accuracy : 0.629 to ./model.pth
EPOCH 7/10  train_loss=0.943

In [41]:
inference()

Starting video inference...
Processed 100 frames. Stopping for demonstration.
Saving processed video to /content/output_emotion_video.mp4
Displaying the processed video:
